In [1]:
import csv
import re

# Nombres de archivos
input_file = '../datasets/contactos.txt'
output_file = '../datasets/backup_pandadoc_completo.csv'

# Función para detectar si una línea es un email
def is_email(line):
    return re.search(r'[\w\.-]+@[\w\.-]+\.\w+', line)

# Función para detectar si una línea son iniciales (ej: "DA", "3L", "A.")
def is_initials(line):
    # Longitud corta y (es mayúscula o empieza con dígito o termina con punto)
    return len(line) < 5 and (line.isupper() or line[0].isdigit() or line.endswith('.'))

def clean_line(text):
    # Elimina las marcas que aparecieron en tu archivo
    text = re.sub(r'\\', '', text)
    return text.strip()

contacts = []
buffer = []

try:
    with open(input_file, 'r', encoding='utf-8') as f:
        # Leemos línea por línea limpiando basura
        lines = [clean_line(line) for line in f]
        # Filtramos líneas vacías
        lines = [line for line in lines if line]

    for i, line in enumerate(lines):
        if is_email(line):
            email = line
            name = ""
            
            # Analizamos el buffer (líneas anteriores) para encontrar el Nombre
            if buffer:
                prev_1 = buffer[-1] # Línea inmediatamente anterior
                prev_2 = buffer[-2] if len(buffer) > 1 else None # Línea anterior a esa
                
                # Lógica para identificar el nombre basado en la estructura
                # Caso 1: Iniciales -> Nombre -> Email
                if prev_2 and is_initials(prev_2):
                    name = prev_1
                # Caso 2: Iniciales -> Nombre -> Empresa -> Email (Tomamos Nombre, ignoramos empresa)
                elif prev_2 and not is_initials(prev_2) and not is_email(prev_2):
                    name = prev_2
                # Caso 3: Nombre -> Email (Sin iniciales)
                elif not is_initials(prev_1) and not is_email(prev_1):
                    name = prev_1
                # Caso 4: El nombre no está claro, usamos la parte del email antes del @
                else:
                    name = email.split('@')[0]
            else:
                # Si no hay buffer, usamos el email como nombre
                name = email.split('@')[0]

            # Evitar duplicados exactos si el archivo tiene basura repetida
            if not contacts or contacts[-1][1] != email:
                contacts.append([name, email])
            
            buffer = [] # Reiniciamos el buffer tras encontrar un contacto
        else:
            buffer.append(line)

    # Escribir el resultado a CSV
    with open(output_file, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Name', 'Email']) # Encabezados
        writer.writerows(contacts)

    print(f"✅ ¡Éxito! Se han procesado {len(contacts)} contactos.")
    print(f"📂 Archivo guardado como: {output_file}")

except FileNotFoundError:
    print(f"❌ Error: No se encuentra el archivo '{input_file}'. Asegúrate de que esté en la misma carpeta.")

✅ ¡Éxito! Se han procesado 2166 contactos.
📂 Archivo guardado como: ../datasets/backup_pandadoc_completo.csv
